In [1]:
# pacotes usados neste notebook
import pandas as pd

In [18]:
df = pd.read_csv('./datasets/classification_results_trial_0001.csv')

In [19]:
df


,image_path,real_class,predicted_class,prob_benign,prob_malign
0,image_001.jpg,malign,malign,0.031429,0.968571
1,image_002.jpg,benign,benign,0.636410,0.363590
2,image_003.jpg,malign,malign,0.314356,0.685644
3,image_004.jpg,benign,benign,0.508571,0.491429
4,image_005.jpg,benign,benign,0.907566,0.092434
...,...,...,...,...,...
95,image_096.jpg,malign,malign,0.349210,0.650790
96,image_097.jpg,benign,benign,0.725956,0.274044
97,image_098.jpg,benign,benign,0.897110,0.102890
98,image_099.jpg,benign,benign,0.887086,0.112914


# 1. Quantas imagens são “benign” e “malign” em real_class? 

In [20]:
contagem = df['real_class'].value_counts()
print(f"Contagem Real:\n{contagem}\n")

Contagem Real:
real_class
malign    55
benign    45
Name: count, dtype: int64



# 2. Identifique em quais imagens o modelo errou a predição. 

In [24]:
erros = df[df['real_class'] != df['predicted_class']]

print(erros[['image_path', 'real_class', 'predicted_class']])
print(f"Total de erros: {len(erros)}")

       image_path real_class predicted_class
20  image_021.jpg     malign          benign
26  image_027.jpg     malign          benign
27  image_028.jpg     malign          benign
41  image_042.jpg     benign          malign
46  image_047.jpg     malign          benign
65  image_066.jpg     malign          benign
67  image_068.jpg     benign          malign
69  image_070.jpg     malign          benign
70  image_071.jpg     malign          benign
76  image_077.jpg     malign          benign
80  image_081.jpg     benign          malign
91  image_092.jpg     malign          benign
Total de erros: 12


# 3. Verifique se o modelo estava confiante mesmo quando errou. 

In [26]:
erros_confiantes = erros[(erros['prob_benign'] > 0.8) | (erros['prob_malign'] > 0.8)]
print(f"Erros com alta confiança (>80%):\n{erros_confiantes}\n")

Erros com alta confiança (>80%):
       image_path real_class predicted_class  prob_benign  prob_malign  \
20  image_021.jpg     malign          benign     0.807440     0.192560   
26  image_027.jpg     malign          benign     0.818015     0.181985   
27  image_028.jpg     malign          benign     0.860731     0.139269   
65  image_066.jpg     malign          benign     0.835302     0.164698   
67  image_068.jpg     benign          malign     0.186519     0.813481   
91  image_092.jpg     malign          benign     0.897216     0.102784   

    confidence  
20    0.807440  
26    0.818015  
27    0.860731  
65    0.835302  
67    0.813481  
91    0.897216  



# 4. Calcule: 
## a. TP (real malign, previsto malign) 
## b. TN (real benign, previsto benign) 
## c. FP (real benign, previsto malign) 
## d. FN (real malign, previsto benign)

In [27]:
tp = len(df[(df['real_class'] == 'malign') & (df['predicted_class'] == 'malign')])
tn = len(df[(df['real_class'] == 'benign') & (df['predicted_class'] == 'benign')])
fp = len(df[(df['real_class'] == 'benign') & (df['predicted_class'] == 'malign')])
fn = len(df[(df['real_class'] == 'malign') & (df['predicted_class'] == 'benign')])

In [37]:
print(f"a) TP: {tp} \nb) TN: {tn} \nc) FP: {fp} \nd) FN: {fn}\n")

a) TP: 46 
b) TN: 42 
c) FP: 3 
d) FN: 9



# 5. Calcule: 
### a. Acurácia: (TP+TN)/(TP+TN+FP+FN) 
### b. Precisão: TP/(TP+FP) 
### c. Recall: TP/(TP+FN) 
### d. Especificidade: TN/(TN+FP)

In [33]:
acuracia = (tp + tn) / (tp + tn + fp + fn)
precisao = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
especificidade = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"Acurácia: {acuracia:.2f}")
print(f"Precisão: {precisao:.2f}")
print(f"Recall: {recall:.2f}")
print(f"Especificidade: {especificidade:.2f}\n")

Acurácia: 0.88
Precisão: 0.94
Recall: 0.84
Especificidade: 0.93



# 6. Encontre as 5 imagens benign com menor prob_benign. O que isso pode indicar? 

In [38]:
min_benign_prob = (
    df[df['real_class'] == 'benign']
    .sort_values(by='prob_benign')
    .head(5)
)

print("5 imagens benign com menor prob_benign:")
print(min_benign_prob[['image_path', 'prob_benign']])

5 imagens benign com menor prob_benign:
       image_path  prob_benign
67  image_068.jpg     0.186519
41  image_042.jpg     0.251782
80  image_081.jpg     0.341066
47  image_048.jpg     0.502679
3   image_004.jpg     0.508571


Resposta: Indica que o modelo quase as classificou como malign, indicando casos de Falso Positivo em potencial. O modelo está vendo características que ele associa a tumors malignos em imagens que são benignas. Na prática médica, isso gera alarmes falsos e biópsias desnecessárias.

# 7. Encontre as 5 imagens malign com maior prob_benign. O que isso pode indicar? 

In [36]:
max_prob_bening = (
    df[df['real_class'] == 'malign']
    .sort_values(by='prob_benign', ascending=False)
    .head(5)
)

print("5 imagens malign com maior prob_benign:")
print(max_prob_bening[['image_path', 'prob_benign']])

5 imagens malign com maior prob_benign:
       image_path  prob_benign
91  image_092.jpg     0.897216
27  image_028.jpg     0.860731
65  image_066.jpg     0.835302
26  image_027.jpg     0.818015
20  image_021.jpg     0.807440


Resposta: Isso indica os casos mais perigosos: Falsos Negativos. O modelo está muito confiante de que a imagem é benigna, mas ela é maligna. Isso pode indicar que o modelo não captou padrões específicos de malignidade nessas imagens ou que os dados de treino estavam viciados